# 1 - coletor CEMADEN (simula o ESP32)

Consulta a API do CEMADEN PED em intervalos regulares e publica os dados brutos de pluviometria via MQTT.

Na arquitetura final do projeto esse papel é do ESP32 físico. Aqui, um script Python faz exatamente o mesmo trabalho: autentica, puxa os acumulados das estações de Florianópolis e publica no broker público do HiveMQ.

**tópico publicado:** `flood/cemaden/raw`


In [1]:
import os
import json
import time
import requests
import paho.mqtt.client as mqtt
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

cemaden_token = os.getenv("CEMADEN_TOKEN")
cemaden_base = os.getenv("CEMADEN_BASE")
codibge = os.getenv("CEMADEN_CODIBGE")

mqtt_broker = os.getenv("MQTT_BROKER")
mqtt_port = int(os.getenv("MQTT_PORT"))
mqtt_topic = os.getenv("MQTT_TOPIC_RAW")

intervalo_seg = 600  # 10 minutos, respeita o limite de 12 req/min do cemaden


In [2]:
def salvar_json(payload, origem="coletor_cemaden"):
    ts = payload.get("ts_coleta", time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
    pasta = Path("landing") / origem
    pasta.mkdir(parents=True, exist_ok=True)
    arquivo = pasta / f"{ts.replace(':', '-')}.json"
    with open(arquivo, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"[landing] salvo em {arquivo}")

## coleta na api do cemaden

Dois endpoints usados:
- `/pcds-cadastro/dados-cadastrais` — coordenadas e nome de cada estação
- `/pcds-acum/acumulados-recentes` — mm acumulados em janelas de 1h até 120h

O join entre os dois é feito pelo campo `codestacao`.


In [3]:
def buscar_estacoes():
    url = f"{cemaden_base}/pcds-cadastro/dados-cadastrais?codibge={codibge}"
    r = requests.get(url, headers={"token": cemaden_token}, timeout=10)
    r.raise_for_status()
    return r.json()

def buscar_acumulados():
    url = f"{cemaden_base}/pcds-acum/acumulados-recentes?codibge={codibge}"
    r = requests.get(url, headers={"token": cemaden_token}, timeout=10)
    r.raise_for_status()
    return r.json()

In [4]:
def montar_payload(estacoes, acumulados):
    mapa_coords = {}
    for e in estacoes:
        mapa_coords[e["codestacao"]] = {
            "nome": e.get("nome", ""),
            "lat": e.get("latitude"),
            "lon": e.get("longitude"),
        }

    leituras = []
    for a in acumulados:
        cod = a.get("codestacao")
        coords = mapa_coords.get(cod, {})
        leituras.append({
            "codestacao": cod,
            "nome": coords.get("nome", ""),
            "lat": coords.get("lat"),
            "lon": coords.get("lon"),
            "timestamp": a.get("datahora"),
            "chuva_1h": a.get("acc1hr"),
            "chuva_3h": a.get("acc3hr"),
            "chuva_6h": a.get("acc6hr"),
            "chuva_12h": a.get("acc12hr"),
            "chuva_24h": a.get("acc24hr"),
            "chuva_48h": a.get("acc48hr"),
            "chuva_72h": a.get("acc72hr"),
            "chuva_96h": a.get("acc96hr"),
            "chuva_120h": a.get("acc120hr"),
        })

    return {
        "fonte": "cemaden_ped",
        "codibge": codibge,
        "ts_coleta": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "estacoes": leituras,
    }

## publicação mqtt

Conecta no broker público do HiveMQ (sem autenticação) e publica o payload como JSON no tópico configurado.


In [5]:
def publicar(cliente, payload):
    msg = json.dumps(payload, ensure_ascii=False)
    resultado = cliente.publish(mqtt_topic, msg, qos=1)
    resultado.wait_for_publish()
    print(f"[mqtt] publicado {len(payload['estacoes'])} estações em '{mqtt_topic}'")


## loop principal

Roda indefinidamente, coletando e publicando a cada `intervalo_seg` segundos. Erros HTTP são logados sem derrubar o loop.


In [ ]:
cliente = mqtt.Client(client_id="esp32-simulado-cemaden", protocol=mqtt.MQTTv311)
cliente.connect(mqtt_broker, mqtt_port, keepalive=60)
cliente.loop_start()

print(f"[info] conectado ao broker {mqtt_broker}")
print(f"[info] coletando a cada {intervalo_seg}s. Ctrl+C para parar.\n")

while True:
    try:
        print("[cemaden] buscando estações...")
        estacoes = buscar_estacoes()
        acumulados = buscar_acumulados()
        payload = montar_payload(estacoes, acumulados)

        salvar_json({'estacoes': estacoes, 'acumulados': acumulados}, origem="coletor_cemaden/raw")
        salvar_json(payload, origem="coletor_cemaden/treated")

        print(f"[cemaden] {len(payload['estacoes'])} estações coletadas.")
        publicar(cliente, payload)

    except requests.HTTPError as e:
        print(f"[erro] HTTP {e.response.status_code}: {e.response.text[:200]}")
    except Exception as e:
        print(f"[erro] {e}")

    print(f"[info] aguardando {intervalo_seg}s...\n")
    time.sleep(intervalo_seg)


C:\Users\edupc\AppData\Local\Temp\ipykernel_23532\950931154.py:1: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  cliente = mqtt.Client(client_id="esp32-simulado-cemaden", protocol=mqtt.MQTTv311)


[info] conectado ao broker localhost
[info] coletando a cada 600s. Ctrl+C para parar.

[cemaden] buscando estações...
[landing] salvo em landing\coletor_cemaden\raw\2026-06-15T21-41-41Z.json
[landing] salvo em landing\coletor_cemaden\treated\2026-06-15T21-41-41Z.json
[cemaden] 5 estações coletadas.
[mqtt] publicado 5 estações em 'flood/cemaden/raw'
[info] aguardando 600s...

[cemaden] buscando estações...
[landing] salvo em landing\coletor_cemaden\raw\2026-06-15T21-51-43Z.json
[landing] salvo em landing\coletor_cemaden\treated\2026-06-15T21-51-43Z.json
[cemaden] 5 estações coletadas.
[mqtt] publicado 5 estações em 'flood/cemaden/raw'
[info] aguardando 600s...

